# MMIP HW5

## 0. 環境與匯入

In [1]:
# 只需第一次執行（Colab / 新環境）：安裝相依套件
# %pip install -r requirements.txt

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import argparse
import sys
import time
import traceback
from pathlib import Path

from IPython.display import display

BASE_DIR = Path.cwd().resolve()          # notebook 所在目錄 = 專案根目錄
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.utils import ensure_dir, get_logger, resolve_device

logger = get_logger()
print("專案根目錄：", BASE_DIR)

專案根目錄： /root/MMIP/MMIP/HW5


## 1. 參數設定

In [ ]:
def parse_args(argv=None) -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Multimodal Deep Learning Assignment (Quiz 1-3)")
    p.add_argument("--task", choices=["all", "quiz1", "quiz2", "quiz3"], default="all", help="要執行的任務")
    p.add_argument("--device", default="auto", help="auto / cpu / cuda / cuda:0 / mps（不可用時自動退回 CPU）")
    p.add_argument("--data_root", default=str(BASE_DIR / "dataset"), help="資料集根目錄")
    p.add_argument("--result_root", default=str(BASE_DIR / "result"), help="結果輸出根目錄")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--num_workers", type=int, default=0, help="DataLoader 執行緒數；Windows 若出錯請改 0")
    p.add_argument("--quick", action="store_true", help="極小子集冒煙測試（不代表正式結果）")

    g1 = p.add_argument_group("Quiz 1")
    g1.add_argument("--quiz1_num_query", type=int, default=10000)
    g1.add_argument("--quiz1_gallery_size", type=int, default=None, help="Gallery 張數；預設用 train 全部（50000），想加快可設 10000")
    g1.add_argument("--quiz1_download", action="store_true", help="本地找不到資料時才允許下載（預設不下載）")
    g1.add_argument("--quiz1_batch_size", type=int, default=128)

    g2 = p.add_argument_group("Quiz 2")
    g2.add_argument("--quiz2_combos", nargs="+", default=["A", "B"], help="A=ResNet50+BERT, B=ViT+BERT, C=ResNet50+RoBERTa")
    g2.add_argument("--quiz2_epochs", type=int, default=5)
    g2.add_argument("--quiz2_batch_size", type=int, default=32)
    g2.add_argument("--quiz2_lr_head", type=float, default=1e-3)
    g2.add_argument("--quiz2_lr_backbone", type=float, default=2e-5)
    g2.add_argument("--quiz2_finetune", action="store_true", help="解凍並端到端微調骨幹（較慢、需較多顯存）")

    g3 = p.add_argument_group("Quiz 3")
    g3.add_argument("--quiz3_epochs", type=int, default=3)
    g3.add_argument("--quiz3_batch_size", type=int, default=32)
    g3.add_argument("--quiz3_prefix_len", type=int, default=10)
    g3.add_argument("--quiz3_max_train_images", type=int, default=None)
    g3.add_argument("--quiz3_eval_images", type=int, default=1000)
    g3.add_argument("--quiz3_week4_baseline", default=None, help="Week 4 baseline 指標 JSON 路徑")
    return p.parse_args(argv)


# 在這裡修改參數（等同 python main.py --task all --device cuda）
args = parse_args([
    "--task", "all",
    "--device", "cuda",
    # "--quick",                 
    # "--quiz2_combos", "A", "B", "C",
    # "--quiz2_epochs", "8",
    # "--quiz3_epochs", "5",
])
print(args)

Namespace(task='all', device='cuda', data_root='/root/MMIP/MMIP/HW5/dataset', result_root='/root/MMIP/MMIP/HW5/result', seed=42, num_workers=0, quick=False, quiz1_num_query=10000, quiz1_gallery_size=None, quiz1_download=False, quiz1_batch_size=128, quiz2_combos=['A', 'B'], quiz2_epochs=5, quiz2_batch_size=32, quiz2_lr_head=0.001, quiz2_lr_backbone=2e-05, quiz2_finetune=False, quiz3_epochs=3, quiz3_batch_size=32, quiz3_prefix_len=10, quiz3_max_train_images=None, quiz3_eval_images=1000, quiz3_week4_baseline=None)


## 2. 建立資料夾、選擇裝置、定義執行器

In [4]:
data_root, result_root = Path(args.data_root).resolve(), Path(args.result_root).resolve()
for q in ("quiz1", "quiz2", "quiz3"):          # 自動建立所有資料 / 結果資料夾
    ensure_dir(data_root / q)
    ensure_dir(result_root / q)

device = resolve_device(args.device)
selected = ["quiz1", "quiz2", "quiz3"] if args.task == "all" else [args.task]
status = {}
logger.info("使用裝置：%s | 任務：%s", device, selected)


def run_quiz1():
    from src import quiz1_retrieval
    return quiz1_retrieval.run(
        data_dir=data_root / "quiz1", result_dir=result_root / "quiz1", device=device, seed=args.seed,
        num_query=args.quiz1_num_query, gallery_size=args.quiz1_gallery_size,
        batch_size=args.quiz1_batch_size, num_workers=args.num_workers,
        allow_download=args.quiz1_download,
    )


def run_quiz2():
    from src import quiz2_multimodal
    return quiz2_multimodal.run(
        data_dir=data_root / "quiz2", result_dir=result_root / "quiz2", device=device, seed=args.seed,
        combos=args.quiz2_combos, epochs=args.quiz2_epochs, batch_size=args.quiz2_batch_size,
        lr_head=args.quiz2_lr_head, lr_backbone=args.quiz2_lr_backbone, finetune=args.quiz2_finetune,
        num_workers=args.num_workers, quick=args.quick,
    )


def run_quiz3():
    from src import quiz3_captioning
    return quiz3_captioning.run(
        data_dir=data_root / "quiz3", result_dir=result_root / "quiz3", device=device, seed=args.seed,
        epochs=args.quiz3_epochs, batch_size=args.quiz3_batch_size, prefix_len=args.quiz3_prefix_len,
        num_workers=args.num_workers, max_train_images=args.quiz3_max_train_images,
        num_eval_images=args.quiz3_eval_images,
        week4_baseline=Path(args.quiz3_week4_baseline) if args.quiz3_week4_baseline else None,
        quick=args.quick,
    )


def safe_run(name, fn):
    """執行單一任務並捕捉例外：失敗只印出原因，不影響其他 Quiz。"""
    logger.info("######## 開始 %s ########", name)
    t0 = time.time()
    try:
        result = fn()
        status[name] = f"成功（{time.time() - t0:.0f}s）"
        return result
    except FileNotFoundError as e:
        logger.error("%s 資料缺失：%s", name, e)
        status[name] = "失敗：資料缺失"
    except KeyboardInterrupt:
        logger.error("使用者中斷。")
        status[name] = "中斷"
    except Exception as e:
        logger.error("%s 執行失敗：%s\n%s", name, e, traceback.format_exc())
        status[name] = f"失敗：{type(e).__name__}"
    return None

[23:59:33] INFO 使用裝置：cuda | 任務：['quiz1', 'quiz2', 'quiz3']


## 3. Quiz 1：影像檢索（ResNet-50 vs ViT-B/16）

In [ ]:
df1 = safe_run("quiz1", run_quiz1) if "quiz1" in selected else None

## 4. Quiz 2：多模態分類（Hateful Memes）

In [ ]:
df2 = safe_run("quiz2", run_quiz2) if "quiz2" in selected else None

[23:59:34] INFO ######## 開始 quiz2 ########
[23:59:34] INFO 資料量：train=7650, val=850, test(dev)=500


/root/MMIP/MMIP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[23:59:35] INFO ===== 組合 A: ResNet-50 + BERT =====


## 5. Quiz 3：CLIP Image Captioning

In [ ]:
df3 = safe_run("quiz3", run_quiz3) if "quiz3" in selected else None